In [27]:
import pandas as pd
import geopandas as gpd
import html
pd.set_option('display.max_columns', None)

In [28]:
usda = pd.read_csv("../data/raw/usda-plants 8-1-2023(in).csv")
trees = gpd.read_file("../data/processed/philly_trees.geojson")

In [29]:
species = trees.drop_duplicates(subset=["tree_name"])


In [30]:
# cleaning USDA of html entities
usda["ScientificName_clean"] = (
    usda["ScientificName"]
    .fillna("")
    .map(html.unescape)                          # &lt;i&gt; → <i>
    .str.replace(r"<[^>]+>", "", regex=True)     # strip <i>, </i>, any other tags
    .str.split().str[:2].str.join(" ")           # keep just "Genus species", drop authority
    .str.strip()
)

# normalize cases for common and scientific name for both datasets
usda["ScientificName_clean"] = usda["ScientificName_clean"].str.lower()
usda["CommonName"] = usda["CommonName"].str.lower()
species["scientific_name"] = species["scientific_name"].str.lower()
species["common_name"] = species["common_name"].str.lower()


/var/folders/rs/nkr8ggln0cxg2bh1hwm7mvbm0000gp/T/ipykernel_98353/3455774084.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  usda["ScientificName_clean"] = (


In [39]:
matches = species.loc[(species["scientific_name"].isin(usda["ScientificName_clean"]))
                      | (species["common_name"].isin(usda["CommonName"]))]
misses = species.loc[~((species["scientific_name"].isin(usda["ScientificName_clean"]))
                      | (species["common_name"].isin(usda["CommonName"])))]

In [42]:
misses

,objectid,tree_name,scientific_name,common_name,Genus,Species,geometry
1,2,Acer palmatum - japanese maple,acer palmatum,japanese maple,Acer,palmatum,POINT (-75.21053 39.98374)
7,9,Cornus kousa - kousa dogwood,cornus kousa,kousa dogwood,Cornus,kousa,POINT (-75.21003 39.98398)
8,10,Amelanchier species - other serviceberry,amelanchier species,other serviceberry,Amelanchier,species,POINT (-75.20996 39.98388)
34,36,Prunus sargentii - sargent cherry,prunus sargentii,sargent cherry,Prunus,sargentii,POINT (-75.12035 40.02201)
35,37,Gleditsia triacanthos inermis - thornless hone...,gleditsia triacanthos inermis,thornless honeylocust,Gleditsia,triacanthos,POINT (-75.12101 40.02165)
...,...,...,...,...,...,...,...
68827,69306,Malus species - sugar tyme crabapple,malus species,sugar tyme crabapple,Malus,species,POINT (-75.15918 39.96229)
78103,78656,Rosa species - other rose,rosa species,other rose,Rosa,species,POINT (-75.14111 39.99438)
140692,142126,Acer henryii – henrys maple,acer henryii,henrys maple,Acer,henryii,POINT (-75.12051 40.04971)
147091,148593,Malus species - indian summer crabapple,malus species,indian summer crabapple,Malus,species,POINT (-75.21815 39.94158)
